In [39]:
from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.response.FullDetectorResponse import FullDetectorResponse
from cosipy.util import fetch_wasabi_file
from histpy import Histogram

from scoords import SpacecraftFrame

from astropy.time import Time
import astropy.units as u

SED_KEV_TO_ERG = u.keV.to(u.erg)
KEV_TO_MEV = u.keV.to(u.MeV)
from astropy.coordinates import SkyCoord, Galactic

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from threeML import *
from threeML.io.package_data import get_path_of_data_file
from threeML.io.logging import silence_console_log
from astromodels import Parameter
from threeML.minimizer.minimization import CannotComputeCovariance

from jupyterthemes import jtplot
jtplot.style(context="talk", fscale=1, ticks=True, grid=False)
set_threeML_style()
silence_warnings()

from scipy.integrate import quad

import matplotlib.ticker as mticker

from pathlib import Path

import os

%matplotlib inline

In [40]:
data_path = Path("/Users/parshadkp/Software/COSI_Data/")

In [41]:
from astropy import units as u
from astropy.coordinates import SkyCoord
from cosipy.event_selection import GoodTimeInterval
from agn_cosi_fit_utils import open_spacecraft_history, scale_spacecraft_livetime

orientation_path = "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits"
source_coord = SkyCoord(l=155.077, b=75.063, frame="galactic", unit="deg")
fov_cut = 60 * u.deg

full_sc_orientation = open_spacecraft_history(orientation_path)
source_gti = GoodTimeInterval.from_pointing_cut(
    source_coord,
    full_sc_orientation,
    fov_cut,
    earth_occ=False,
)
sc_orientation = full_sc_orientation.apply_gti(source_gti)

print(f"NGC 4151 FOV cut: {fov_cut.to_value(u.deg):.0f} deg")
print(f"Selected livetime: {sc_orientation.cumulative_livetime().to_value(u.s):,.1f} s")

NGC 4151 FOV cut: 60 deg
Selected livetime: 980,415.0 s


In [42]:
dr = "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5"

In [43]:
multiplier_4151 = 1
multiplier_1068 = 1

exposure_4151 = multiplier_4151 * 3

# Scale count histograms and response livetime together; keep source flux intrinsic.
sc_orientation = scale_spacecraft_livetime(sc_orientation, multiplier_4151)
exposure_1068 = multiplier_1068 * 3

# Source Model Plots

## Cutoff Power Law (Thermal)

#### NGC 4151 (200 keV)

In [44]:
K_inj = 0.15 / u.cm / u.cm / u.s / u.keV
piv_inj = 1. * u.keV
xc_inj = 200. * u.keV
index_inj = -1.75

spectrum_inj_ec200 = Cutoff_powerlaw()

spectrum_inj_ec200.K.value = K_inj.value
spectrum_inj_ec200.piv.value = piv_inj.value
spectrum_inj_ec200.xc.value = xc_inj.value
spectrum_inj_ec200.index.value = index_inj

spectrum_inj_ec200.K.unit = K_inj.unit
spectrum_inj_ec200.piv.unit = piv_inj.unit
spectrum_inj_ec200.xc.unit = xc_inj.unit

#### NGC 4151 (1000 keV)

In [45]:
K_inj = 0.15 / u.cm / u.cm / u.s / u.keV
piv_inj = 1. * u.keV
xc_inj = 1000. * u.keV
index_inj = -1.75

spectrum_inj_ec1000 = Cutoff_powerlaw()

spectrum_inj_ec1000.K.value = K_inj.value
spectrum_inj_ec1000.piv.value = piv_inj.value
spectrum_inj_ec1000.xc.value = xc_inj.value
spectrum_inj_ec1000.index.value = index_inj

spectrum_inj_ec1000.K.unit = K_inj.unit
spectrum_inj_ec1000.piv.unit = piv_inj.unit
spectrum_inj_ec1000.xc.unit = xc_inj.unit

# Power law tail (Non-thermal)

#### NGC 4151 (200 keV)

In [46]:
# K_inj = 0.1*spectrum_inj_ec200.evaluate_at(1000) / u.cm / u.cm / u.s / u.keV
# piv_inj = 1000. * u.keV

def cutoff_powerlaw_k_at_pivot(shape, pivot_value):
    return float(shape.K.value * (pivot_value / shape.piv.value) ** shape.index.value)

K_inj = 0.65*spectrum_inj_ec200.evaluate_at(200) / u.cm / u.cm / u.s / u.keV
piv_inj = 200. * u.keV
index_inj = -2.8

spectrum_inj_ec200_PL = Powerlaw()

spectrum_inj_ec200_PL.K.value = K_inj.value
spectrum_inj_ec200_PL.piv.value = piv_inj.value
spectrum_inj_ec200_PL.index.value = index_inj

spectrum_inj_ec200_PL.K.unit = K_inj.unit
spectrum_inj_ec200_PL.piv.unit = piv_inj.unit

spectrum_inj_ec200_total = spectrum_inj_ec200 + spectrum_inj_ec200_PL

print("K_inj: ", K_inj)
print(spectrum_inj_ec200.evaluate_at(200))
print(spectrum_inj_ec200_PL.evaluate_at(200))
print("="*40)
linking_ratio_ec200 = spectrum_inj_ec200_PL.K.value / cutoff_powerlaw_k_at_pivot(
    spectrum_inj_ec200,
    spectrum_inj_ec200_PL.piv.value,
)
print("Linking K ratio", linking_ratio_ec200)
print("Flux ratio at 200 keV", spectrum_inj_ec200_PL.evaluate_at(200)/spectrum_inj_ec200.evaluate_at(200))
print("="*40)
flux_th, _ = quad(spectrum_inj_ec200.evaluate_at, 200.0, 5000.0)
flux_nth, _ = quad(spectrum_inj_ec200_PL.evaluate_at, 200.0, 5000.0)
print("Non-thermal flux: ", flux_nth)
print("Total flux: ", flux_th + flux_nth)
print("Thermal - Non-thermal Flux Ratio: ", flux_th/flux_nth)
print("Non-thermal percentage: ", flux_nth/(flux_th + flux_nth))

K_inj:  3.3721558756085362e-06 1 / (keV s cm2)
5.187932116320825e-06
3.3721558756085362e-06
Linking K ratio 0.23912163676143755
Flux ratio at 200 keV 0.65
Non-thermal flux:  0.00037354275634502636
Total flux:  0.0008309220159224175
Thermal - Non-thermal Flux Ratio:  1.2244361637545138
Non-thermal percentage:  0.4495521230477347


#### NGC 4151 (1000 keV)

In [47]:
# K_inj = (spectrum_inj_ec1000.evaluate_at(3000)) / u.cm / u.cm / u.s / u.keV
K_inj = 0.36*spectrum_inj_ec1000.evaluate_at(200) / u.cm / u.cm / u.s / u.keV
piv_inj = 200. * u.keV
index_inj = -2.8

spectrum_inj_ec1000_PL = Powerlaw()

spectrum_inj_ec1000_PL.K.value = K_inj.value
spectrum_inj_ec1000_PL.piv.value = piv_inj.value
spectrum_inj_ec1000_PL.index.value = index_inj

spectrum_inj_ec1000_PL.K.unit = K_inj.unit
spectrum_inj_ec1000_PL.piv.unit = piv_inj.unit

spectrum_inj_ec1000_total = spectrum_inj_ec1000 + spectrum_inj_ec1000_PL

print("K_inj: ", K_inj)
print(spectrum_inj_ec1000.evaluate_at(200))
print(spectrum_inj_ec1000_PL.evaluate_at(200))
print("="*40)
linking_ratio_ec1000 = spectrum_inj_ec1000_PL.K.value / cutoff_powerlaw_k_at_pivot(
    spectrum_inj_ec1000,
    spectrum_inj_ec1000_PL.piv.value,
)
print("Linking K ratio", linking_ratio_ec1000)
print("Flux ratio at 200 keV", spectrum_inj_ec1000_PL.evaluate_at(200)/spectrum_inj_ec1000.evaluate_at(200))
print("="*40)
flux_th, _ = quad(spectrum_inj_ec1000.evaluate_at, 200.0, 5000.0)
flux_nth, _ = quad(spectrum_inj_ec1000_PL.evaluate_at, 200.0, 5000.0)
print("Non-thermal flux: ", flux_nth)
print("Total flux: ", flux_th + flux_nth)
print("Thermal - Non-thermal Flux Ratio: ", flux_th/flux_nth)
print("Non-thermal percentage: ", flux_nth/(flux_th + flux_nth))

K_inj:  4.156543893280518e-06 1 / (keV s cm2)
1.1545955259112548e-05
4.156543893280513e-06
Linking K ratio 0.2947430711080734
Flux ratio at 200 keV 0.35999999999999965
Non-thermal flux:  0.0004604315221593666
Total flux:  0.002355963865192861
Thermal - Non-thermal Flux Ratio:  4.116860492400007
Non-thermal percentage:  0.19543233619233602


# Spectral Fitting

In [48]:
## 24-months
# NGC4151_ec200 = Histogram.open(
#     data_path
#     / "AGN_Data/GammaRay/Paper_Models/"
#       "NGC4151_ec_200_-2.8_DC4_COSI_cpl_pl_60_fovCut_normNT_0p10.hdf5"
# ) * multiplier_4151

## 3-months
NGC4151_ec200 = Histogram.open(
    data_path
    / "AGN_Data/GammaRay/Paper_Models/"
      "NGC4151_ec_200_-2.8_DC4_COSI_cpl_pl_60_fovCut_normNT_0p65.hdf5"
) * multiplier_4151

bkg = Histogram.open(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/"
    "COSI/Software_Files/DC4_Files/Background/"
    "Total_DC4_BG_3months_binned_data_filtered_with_SAAcut_withSAAbck_"
    "NGC4151_60deg_fov_cut.hdf5"
) * multiplier_4151

# Collapse the background time axis.
bkg = bkg.project("Em", "Phi", "PsiChi")

# Match metadata, following the source-injector branch workflow.
NGC4151_ec200.axes["Em"].axis_scale = bkg.axes["Em"].axis_scale
NGC4151_ec200 = NGC4151_ec200.to(
    unit=bkg.unit,
    update=False,
)

NGC4151_ec200_bkg = NGC4151_ec200 + bkg

In [49]:
# ## 24-months
# NGC4151_ec1000 = Histogram.open(data_path/"AGN_Data/GammaRay/Paper_Models/NGC4151_ec_1000_-2.8_DC4_COSI_cpl_pl_60_fovCut_normNT_0p12.hdf5")*multiplier_4151

## 3-months
NGC4151_ec1000 = Histogram.open(data_path/"AGN_Data/GammaRay/Paper_Models/NGC4151_ec_1000_-2.8_DC4_COSI_cpl_pl_60_fovCut_normNT_0p36.hdf5")*multiplier_4151

# Match metadata, following the source-injector branch workflow.
NGC4151_ec1000.axes["Em"].axis_scale = bkg.axes["Em"].axis_scale
NGC4151_ec1000 = NGC4151_ec1000.to(
    unit=bkg.unit,
    update=False,
)

NGC4151_ec1000_bkg = NGC4151_ec1000 + bkg

## Perform spectral fit

Set background parameter, which is used to fit the amplitude of the background, and instantiate the COSI 3ML plugin

In [50]:
from agn_cosi_fit_utils import COSIPlugin, EnergyRangeCOSIPlugin, make_cosi_background_parameter
bkg_par = Parameter("background_cosi",                                         # background parameter
                     1,                                                        # initial value of parameter
                     min_value=0,                                              # minimum value of parameter
                     max_value=5,                                              # maximum value of parameter
                     delta=0.05,                                               # initial step used by fitting engine
                     desc="Background parameter for cosi")

bkg_par_ec1000 = Parameter("background_cosi_ec1000",                           # background parameter
                     1,                                                        # initial value of parameter
                     min_value=0,                                              # minimum value of parameter
                     max_value=5,                                              # maximum value of parameter
                     delta=0.05,                                               # initial step used by fitting engine
                     desc="Background parameter for cosi")

cosi = COSIPlugin("cosi",                                                        # COSI 3ML plugin
                 dr = dr,                                                      # detector response
                 data = NGC4151_ec200_bkg.project('Em', 'Phi', 'PsiChi'),   # data (source+background)
                 bkg = bkg.project('Em', 'Phi', 'PsiChi'),         # background model 
                 sc_orientation = sc_orientation,                              # spacecraft orientation
                 nuisance_param = bkg_par,                                     # background parameter
                 earth_occ = True)                                             # Option to account for Earth occultation

cosi_ec1000 = COSIPlugin("cosi",                                                        # COSI 3ML plugin
                 dr = dr,                                                      # detector response
                 data = NGC4151_ec1000_bkg.project('Em', 'Phi', 'PsiChi'),   # data (source+background)
                 bkg = bkg.project('Em', 'Phi', 'PsiChi'),         # background model 
                 sc_orientation = sc_orientation,                              # spacecraft orientation
                 nuisance_param = bkg_par_ec1000,                                     # background parameter
                 earth_occ = True)                                             # Option to account for Earth occultation

### Powerlaw with energy cutoff fit

In [51]:
l=155.07
b=75.06

# Give it some harsher initial guesses
K = 1e-5 / u.cm / u.cm / u.s / u.keV
piv = 200. * u.keV
xc = 100. * u.keV
index = -1.75

spectrum_cpl = Cutoff_powerlaw()

spectrum_cpl.K.value = K.value
spectrum_cpl.piv.value = piv.value
spectrum_cpl.xc.value = xc.value
spectrum_cpl.index.value = index
spectrum_cpl.index.fix = True

# Harsher Parameters
spectrum_cpl.K.min_value = 1e-8
spectrum_cpl.K.max_value = 1e-2
spectrum_cpl.xc.min_value = 100 # keep these relatively the same
spectrum_cpl.xc.max_value = 10000 # change to 1000
# spectrum_cpl.index.min_value = -3 # change to -3.5 to sample lower, larger values (-5, 5)
# spectrum_cpl.index.max_value = 1

# spectrum_cpl.K.delta = 0.05
# spectrum_cpl.xc.delta = 10
# spectrum_cpl.index.delta = 0.15

spectrum_cpl.K.unit = K.unit
spectrum_cpl.piv.unit = piv.unit
spectrum_cpl.xc.unit = xc.unit



In [52]:
l=155.07
b=75.06

# Give it some harsher initial guesses
K = 1e-5 / u.cm / u.cm / u.s / u.keV
piv = 200. * u.keV
xc = 1000. * u.keV
index = -1.75

spectrum_cpl_ec1000 = Cutoff_powerlaw()

spectrum_cpl_ec1000.K.value = K.value
spectrum_cpl_ec1000.piv.value = piv.value
spectrum_cpl_ec1000.xc.value = xc.value
spectrum_cpl_ec1000.index.value = index
spectrum_cpl_ec1000.index.fix = True

# Harsher Parameters
spectrum_cpl_ec1000.K.min_value = 1e-8
spectrum_cpl_ec1000.K.max_value = 1e-2
spectrum_cpl_ec1000.xc.min_value = 100 # keep these relatively the same
spectrum_cpl_ec1000.xc.max_value = 10000 # change to 1000
# spectrum_cpl_ec1000.index.min_value = -3 # change to -3.5 to sample lower, larger values (-5, 5)
# spectrum_cpl_ec1000.index.max_value = 1

# spectrum_cpl_ec1000.K.delta = 0.05
# spectrum_cpl_ec1000.xc.delta = 10
# spectrum_cpl_ec1000.index.delta = 0.15

spectrum_cpl_ec1000.K.unit = K.unit
spectrum_cpl_ec1000.piv.unit = piv.unit
spectrum_cpl_ec1000.xc.unit = xc.unit



## Thermal + Non-thermal Fit

In [53]:
l=155.07
b=75.06

# Give it some harsher initial guesses
K = 1e-5 / u.cm / u.cm / u.s / u.keV
piv = 200. * u.keV
index = -2.8

spectrum = Powerlaw()

spectrum.K.value = K.value
spectrum.piv.value = piv.value
spectrum.index.value = index
# spectrum.index.fix = True

# Harsher Parameters
spectrum.K.min_value = 1e-8
spectrum.K.max_value = 1e-2
spectrum.index.min_value = -5
spectrum.index.max_value = 1

# spectrum.K.delta = 5
spectrum.index.delta = 0.25

spectrum.K.unit = K.unit
spectrum.piv.unit = piv.unit



In [54]:
l=155.07
b=75.06

# Give it some harsher initial guesses
K = 1e-5 / u.cm / u.cm / u.s / u.keV
piv = 200. * u.keV
index = -2.8

spectrum_ec1000 = Powerlaw()

spectrum_ec1000.K.value = K.value
spectrum_ec1000.piv.value = piv.value
spectrum_ec1000.index.value = index
# spectrum_ec1000.index.fix = True

# Harsher Parameters
spectrum_ec1000.K.min_value = 1e-8
spectrum_ec1000.K.max_value = 1e-2
spectrum_ec1000.index.min_value = -5
spectrum_ec1000.index.max_value = 1

# spectrum_ec1000.K.delta = 5
spectrum_ec1000.index.delta = 0.25

spectrum_ec1000.K.unit = K.unit
spectrum_ec1000.piv.unit = piv.unit



In [55]:
from agn_cosi_fit_utils import COSIPlugin, EnergyRangeCOSIPlugin, make_cosi_background_parameter
# Keep both spectral components in one point source so the response cache
# tracks every component as linked or independently fitted parameters change.
source1 = PointSource(
    "source1",
    l=l,
    b=b,
    spectral_shape=spectrum_cpl + spectrum,
)
source2 = PointSource(
    "source2",
    l=l,
    b=b,
    spectral_shape=spectrum_cpl_ec1000 + spectrum_ec1000,
)

model = Model(source1)
model_ec1000 = Model(source2)

cosi.set_model(model)
cosi_ec1000.set_model(model_ec1000)


def make_cpl_only_source(name, reference_source):
    reference_shape = reference_source.spectrum.main.shape.functions[0]
    cpl_shape = Cutoff_powerlaw()

    for parameter_name in ("K", "piv", "xc", "index"):
        reference_parameter = getattr(reference_shape, parameter_name)
        cpl_parameter = getattr(cpl_shape, parameter_name)
        cpl_parameter.value = reference_parameter.value
        cpl_parameter.fix = reference_parameter.fix

        if reference_parameter.min_value is not None:
            cpl_parameter.min_value = reference_parameter.min_value
        if reference_parameter.max_value is not None:
            cpl_parameter.max_value = reference_parameter.max_value
        if reference_parameter.delta is not None:
            cpl_parameter.delta = reference_parameter.delta
        if reference_parameter.unit is not None:
            cpl_parameter.unit = reference_parameter.unit
    return PointSource(
        name,
        l=l,
        b=b,
        spectral_shape=cpl_shape,
    )


source1_cpl_only = make_cpl_only_source("source1_cpl_only", source1)
source2_cpl_only = make_cpl_only_source("source2_cpl_only", source2)

model_cpl_only = Model(source1_cpl_only)
model_ec1000_cpl_only = Model(source2_cpl_only)

bkg_par_cpl_only = Parameter(
    "background_cosi_cpl_only",
    1,
    min_value=0,
    max_value=5,
    delta=0.05,
    desc="Background parameter for the Cpl-only COSI fit",
)

bkg_par_ec1000_cpl_only = Parameter(
    "background_cosi_ec1000_cpl_only",
    1,
    min_value=0,
    max_value=5,
    delta=0.05,
    desc="Background parameter for the Cpl-only COSI fit",
)

cosi_cpl_only = COSIPlugin(
    "cosi_cpl_only",
    dr=dr,
    data=NGC4151_ec200_bkg.project("Em", "Phi", "PsiChi"),
    bkg=bkg.project("Em", "Phi", "PsiChi"),
    sc_orientation=sc_orientation,
    nuisance_param=bkg_par_cpl_only,
    earth_occ=True,
)

cosi_ec1000_cpl_only = COSIPlugin(
    "cosi_ec1000_cpl_only",
    dr=dr,
    data=NGC4151_ec1000_bkg.project("Em", "Phi", "PsiChi"),
    bkg=bkg.project("Em", "Phi", "PsiChi"),
    sc_orientation=sc_orientation,
    nuisance_param=bkg_par_ec1000_cpl_only,
    earth_occ=True,
)

cosi_cpl_only.set_model(model_cpl_only)
cosi_ec1000_cpl_only.set_model(model_ec1000_cpl_only)


### Joint Fit

In [ ]:
# bat_ec200.use_effective_area_correction(0.02, 1.8)
# plugins = DataList(bat_ec200, cosi)

# bat_ec1000.use_effective_area_correction(0.02, 1.8)
# plugins_ec1000 = DataList(bat_ec1000, cosi_ec1000)

### Only COSI Fit

In [56]:
## Only COSI fit
plugins = DataList(cosi)
plugins_ec1000 = DataList(cosi_ec1000)
plugins_cpl_only = DataList(cosi_cpl_only)
plugins_ec1000_cpl_only = DataList(cosi_ec1000_cpl_only)

### Cpl + Pl vs Cpl-only comparison

In [59]:
ratio_ec200 = linking_ratio_ec200
link_function = Line(a=0.0, b=ratio_ec200)   # tail K = a + b * source1.K
link_function.a.fix = True
link_function.b.min_value = 0.0  # Require a non-negative non-thermal normalization.
# link_function.b.fix = True             # keep ratio fixed at

ratio_ec1000 = linking_ratio_ec1000
link_function_ec1000 = Line(a=0.0, b=ratio_ec1000)   # tail K = a + b * source2.K
link_function_ec1000.a.fix = True
link_function_ec1000.b.min_value = 0.0  # Require a non-negative non-thermal normalization.
# link_function_ec1000.b.fix = True             # keep ratio fixed

model.link(
    model.source1.spectrum.main.composite.K_2,         # dependent
    model.source1.spectrum.main.composite.K_1,  # independent
    link_function,
)
model_ec1000.link(
    model_ec1000.source2.spectrum.main.composite.K_2,         # dependent
    model_ec1000.source2.spectrum.main.composite.K_1,  # independent
    link_function_ec1000,
)

like = JointLikelihood(model, plugins, verbose=False)
like_ec1000 = JointLikelihood(model_ec1000, plugins_ec1000, verbose=False)

like_cpl_only = JointLikelihood(model_cpl_only, plugins_cpl_only, verbose=False)
like_ec1000_cpl_only = JointLikelihood(
    model_ec1000_cpl_only,
    plugins_ec1000_cpl_only,
    verbose=False,
)

result = like.fit()
result_ec1000 = like_ec1000.fit()

result_cpl_only = like_cpl_only.fit()
result_ec1000_cpl_only = like_ec1000_cpl_only.fit()


12:10:36 INFO      set the minimizer to minuit                                             ]8;id=242882;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=88109;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

         INFO      set the minimizer to minuit                                             ]8;id=135583;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=234844;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

         INFO      set the minimizer to minuit                                             ]8;id=332357;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=190817;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

         INFO      set the minimizer to minuit                                             ]8;id=234398;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=216501;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

Best fit values:

...

1.00,-0.81,-0.29
-0.81,1.00,-0.15
-0.29,-0.15,1.00


Values of -log(likelihood) at the minimum:

,-log(likelihood)
cosi_ec1000_cpl_only,-113272392.21112716
total,-113272392.21112716


Values of statistical measures:

,statistical measures
AIC,-226544778.42215016
BIC,-226544747.3795377


In [60]:
from agn_cosi_fit_utils import COSIPlugin, EnergyRangeCOSIPlugin, make_cosi_background_parameter
def make_null_likelihood(data_hist, label):
    bkg_par_null = Parameter(
        f"background_cosi_null_{label}",
        1,
        min_value=0,
        max_value=5,
        delta=0.05,
        desc="Background parameter for the null COSI fit",
    )

    cosi_null = COSIPlugin(
        f"cosi_null_{label}",
        dr=dr,
        data=data_hist.project("Em", "Phi", "PsiChi"),
        bkg=bkg.project("Em", "Phi", "PsiChi"),
        sc_orientation=sc_orientation,
        nuisance_param=bkg_par_null,
        earth_occ=True,
    )

    spectrum_null = Powerlaw()
    spectrum_null.K.value = 1e-30
    spectrum_null.index.value = 1
    spectrum_null.K.fix = True
    spectrum_null.index.fix = True

    source_null = PointSource(
        "source_null",
        l=l,
        b=b,
        spectral_shape=spectrum_null,
    )

    model_null = Model(source_null)
    cosi_null.set_model(model_null)

    plugins_null = DataList(cosi_null)
    like_null = JointLikelihood(model_null, plugins_null, verbose=False)
    like_null.fit()

    return like_null


def get_likelihood_statistic(joint_likelihood):
    statistic_frame = joint_likelihood.results.get_statistic_frame()
    statistic_series = statistic_frame["-log(likelihood)"]

    if "total" in statistic_series.index:
        return float(statistic_series.loc["total"])

    return float(statistic_series.sum())


def detection_ts(null_likelihood, source_likelihood):
    return 2.0 * (
        get_likelihood_statistic(null_likelihood)
        - get_likelihood_statistic(source_likelihood)
    )


def model_improvement_ts(reference_likelihood, test_likelihood):
    return 2.0 * (
        get_likelihood_statistic(reference_likelihood)
        - get_likelihood_statistic(test_likelihood)
    )


like_null_200 = make_null_likelihood(NGC4151_ec200_bkg, "ec200")
like_null_1000 = make_null_likelihood(NGC4151_ec1000_bkg, "ec1000")

TS_cpl_200 = detection_ts(like_null_200, like_cpl_only)
TS_cpl_pl_200 = detection_ts(like_null_200, like)
TS_cpl_pl_vs_cpl_200 = model_improvement_ts(like_cpl_only, like)

TS_cpl_1000 = detection_ts(like_null_1000, like_ec1000_cpl_only)
TS_cpl_pl_1000 = detection_ts(like_null_1000, like_ec1000)
TS_cpl_pl_vs_cpl_1000 = model_improvement_ts(like_ec1000_cpl_only, like_ec1000)

fit_ts_comparison = pd.DataFrame(
    [
        {
            "spectrum": "NGC 4151 Ec=200 keV",
            "TS_Cpl": TS_cpl_200,
            "TS_Cpl_plus_Pl": TS_cpl_pl_200,
            "Delta_TS_Cpl_plus_Pl_vs_Cpl": TS_cpl_pl_vs_cpl_200,
            "Sigma_Cpl": np.sqrt(max(TS_cpl_200, 0.0)),
            "Sigma_Cpl_plus_Pl": np.sqrt(max(TS_cpl_pl_200, 0.0)),
            "Sigma_added_Pl": np.sqrt(max(TS_cpl_pl_vs_cpl_200, 0.0)),
        },
        {
            "spectrum": "NGC 4151 Ec=1000 keV",
            "TS_Cpl": TS_cpl_1000,
            "TS_Cpl_plus_Pl": TS_cpl_pl_1000,
            "Delta_TS_Cpl_plus_Pl_vs_Cpl": TS_cpl_pl_vs_cpl_1000,
            "Sigma_Cpl": np.sqrt(max(TS_cpl_1000, 0.0)),
            "Sigma_Cpl_plus_Pl": np.sqrt(max(TS_cpl_pl_1000, 0.0)),
            "Sigma_added_Pl": np.sqrt(max(TS_cpl_pl_vs_cpl_1000, 0.0)),
        },
    ]
)

display(fit_ts_comparison)

for _, row in fit_ts_comparison.iterrows():
    print(row["spectrum"])
    print(f"  Cpl TS: {row['TS_Cpl']:.3f} ({row['Sigma_Cpl']:.2f} sigma)")
    print(f"  Cpl + Pl TS: {row['TS_Cpl_plus_Pl']:.3f} ({row['Sigma_Cpl_plus_Pl']:.2f} sigma)")
    print(
        "  Added Pl improvement TS: "
        f"{row['Delta_TS_Cpl_plus_Pl_vs_Cpl']:.3f} "
        f"({row['Sigma_added_Pl']:.2f} sigma)"
    )

# Backward-compatible aliases used by the plotting cells below.
TS = TS_cpl_pl_200
TS_ec1000 = TS_cpl_pl_1000

12:10:45 INFO      set the minimizer to minuit                                             ]8;id=299921;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=96144;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

Best fit values:

,result,unit
parameter,,
background_cosi_null_ec200,(2.5674 +/- 0.0005) x 10,Hz


Correlation matrix:

1.00


...

,-log(likelihood)
cosi_null_ec1000,-113270805.02585545
total,-113270805.02585545


Values of statistical measures:

,statistical measures
AIC,-226541608.05169353
BIC,-226541597.7041387


,spectrum,TS_Cpl,TS_Cpl_plus_Pl,Delta_TS_Cpl_plus_Pl_vs_Cpl,Sigma_Cpl,Sigma_Cpl_plus_Pl,Sigma_added_Pl
0,NGC 4151 Ec=200 keV,988.945772,998.432543,9.486771,31.447508,31.597983,3.080060
1,NGC 4151 Ec=1000 keV,3174.370543,3183.717309,9.346766,56.341553,56.424439,3.057248


NGC 4151 Ec=200 keV
  Cpl TS: 988.946 (31.45 sigma)
  Cpl + Pl TS: 998.433 (31.60 sigma)
  Added Pl improvement TS: 9.487 (3.08 sigma)
NGC 4151 Ec=1000 keV
  Cpl TS: 3174.371 (56.34 sigma)
  Cpl + Pl TS: 3183.717 (56.42 sigma)
  Added Pl improvement TS: 9.347 (3.06 sigma)
